# Modulo de Video - Tech Challenge Fase 4

**Roteiro executavel** da parte de video (fisioterapia / postura / marcha) do projeto multimodal.

Este notebook serve para tres coisas: (1) apresentar aos colegas o que e por que foi feito, (2) ser gravado na tela para o video de entrega, e (3) permitir ao professor rodar tudo de cima para baixo.

**O que este modulo entrega:** recebe um video de fisioterapia e produz um alerta padronizado de risco (JSON), no mesmo schema dos outros modulos do grupo, pronto para a fusao multimodal.

> Ambiente: em *Ambiente de execucao > Alterar o tipo de ambiente*, **CPU ja basta**. GPU nao e necessaria.

## Como usar este roteiro

Rode as celulas na ordem, de cima para baixo. O notebook tem duas partes:

1. **Demo do pipeline** (passos 1 a 9): prepara o ambiente e roda a analise em um video, gerando o alerta e o video anotado.
2. **Calibracao** (passos 10 a 14): usa o dataset publico REHAB24-6 para ajustar, com dados reais, os limiares que definem o que e "anomalo".

Cada passo tem um texto curto (o que e por que) seguido da celula que executa.

## 1. Verificar o ambiente

Confere a versao do Python e se ha GPU (nao precisa).

In [ ]:
import sys, platform
print('Python:', sys.version)
print('SO:', platform.platform())
!nvidia-smi -L 2>/dev/null || echo 'Sem GPU (tudo bem, o pipeline roda em CPU).'

## 2. Clonar o repositorio

Baixa o codigo do projeto do GitHub. Se ja estiver clonado, apenas atualiza (`git pull`).

In [ ]:
import os
REPO = '/content/modulo_video'
if not os.path.exists(REPO):
    !git clone https://github.com/marcoswoc/modulo_video.git {REPO}
else:
    !cd {REPO} && git pull
%cd {REPO}

## 3. Instalar as dependencias

Instala MediaPipe (pose), OpenCV (video) e Ultralytics (YOLOv8). Leva 1 a 2 min.

> Os avisos vermelhos de `ERROR: pip's dependency resolver...` sao **esperados** e inofensivos: o Colab ja vem com bibliotecas que pedem versoes diferentes de numpy/protobuf, mas nenhuma delas e usada por este modulo. Se a celula terminar com "Dependencias instaladas.", esta tudo certo.

In [ ]:
!pip install -q -r requirements.txt
# No Colab (Python 3.12) o mediapipe tenta importar o tensorflow pre-instalado e
# quebra por conflito de protobuf. Como este modulo NAO usa tensorflow, removemos
# para o mediapipe carregar normalmente (a pose estimation nao precisa dele).
!pip uninstall -y tensorflow tensorflow-cpu 2>/dev/null
print('Dependencias instaladas.')

## 4. Como funciona o pipeline (visao geral)

O modulo liga 5 etapas em sequencia, mais uma etapa de objetos:

```
video --(1) MediaPipe Pose --> pontos do corpo (postura)
      --(1b) YOLOv8 --> pessoas/eventos (queda, ausencia, contagem)
                |
                v
   (2) biomecanica --> angulos, simetria, estabilidade, velocidade
   (3) regras clinicas --> anomalias
   (4) score de risco --> 0 a 1 + nivel (baixo/moderado/alto)
   (5) relatorio --> alerta JSON padronizado
```

**Modelos e de onde vem:**
- **MediaPipe Pose** para postura e marcha: e exatamente a tecnica ensinada na **Aula 03** (keypoints/landmarks + heuristica sobre eles).
- **YOLOv8** para eventos de seguranca (queda, contagem de pessoas): pedido na **spec** do Tech Challenge. E um aprofundamento alem da aula.

**Por que regras e nao um modelo treinado:** os modelos de visao ja sao pre-treinados; sobre os landmarks aplicamos regras clinicas simples (limiares), faceis de explicar e alinhadas a aula. Esses limiares sao calibrados na parte 2 deste notebook.

## 5. Enviar um video

Escolha um `.mp4` curto (poucos segundos, ate 720p) com a pessoa de **corpo inteiro** visivel. O arquivo vai para `data/entrada/`.

In [ ]:
from google.colab import files
import shutil, os
os.makedirs('data/entrada', exist_ok=True)
enviados = files.upload()
nome_video = list(enviados.keys())[0]
destino = os.path.join('data/entrada', nome_video)
shutil.move(nome_video, destino)
print('Video salvo em:', destino)

## 6. Rodar o pipeline

Gera o **alerta JSON** e o **video anotado** com o esqueleto desenhado.

> Se o video for pesado e demorar, rode a mesma linha com `--sem-objetos` para desligar o YOLOv8.

In [ ]:
PATIENT_ID = 'video_001'
import os
os.makedirs('data/saida', exist_ok=True)
saida_json = 'data/saida/' + PATIENT_ID + '.json'
saida_anotado = 'data/saida/' + PATIENT_ID + '_anotado.mp4'
!python main.py --video "{destino}" --patient-id {PATIENT_ID} --saida "{saida_json}" --anotado "{saida_anotado}"


## 7. Ver o alerta gerado

Este e o produto final do modulo, no schema combinado com o grupo.

In [ ]:
import json
with open(saida_json, encoding='utf-8') as f:
    alerta = json.load(f)
print(json.dumps(alerta, ensure_ascii=False, indent=2))

## 8. Visualizar o video anotado

Mostra o esqueleto desenhado sobre o video (funciona melhor com clipes curtos). Para videos grandes, use o passo 9.

In [ ]:
import os, subprocess
from IPython.display import HTML, display
from base64 import b64encode

if not (os.path.exists(saida_anotado) and os.path.getsize(saida_anotado) > 0):
    print('Video anotado nao foi gerado (verifique o passo 6).')
else:
    # O OpenCV grava em mp4v, que o navegador nao toca inline. Reencoda para
    # H264 (libx264) com o ffmpeg do Colab para exibir dentro do notebook.
    saida_web = saida_anotado.replace('.mp4', '_web.mp4')
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', saida_anotado,
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', saida_web], check=False)
    alvo = saida_web if (os.path.exists(saida_web) and os.path.getsize(saida_web) > 0) else saida_anotado
    dados = open(alvo, 'rb').read()
    if len(dados) > 40_000_000:
        print('Video grande demais para exibir inline. Use o passo 9 para baixar.')
    else:
        url = 'data:video/mp4;base64,' + b64encode(dados).decode()
        display(HTML('<video width=480 controls><source src="' + url + '" type="video/mp4"></video>'))

## 9. Baixar os resultados

Baixa o JSON e o video anotado (uteis no relatorio e na apresentacao).

In [ ]:
from google.colab import files
import os
files.download(saida_json)
if os.path.exists(saida_anotado) and os.path.getsize(saida_anotado) > 0:
    files.download(saida_anotado)

## 10. Calibracao dos limiares com o REHAB24-6

Ate aqui os limiares de "anomalia" eram chutes iniciais. Agora usamos um dataset **publico** para ajusta-los com dados reais.

**Dataset: [REHAB24-6](https://zenodo.org/records/13305826)** (Zenodo, video RGB, download livre, licenca CC-BY-NC). Cada repeticao de exercicio tem rotulo **execucao correta (1) x incorreta (0)**, o que mapeia direto no conceito de anomalia da spec ("movimentos fora do padrao esperado").

**Isto e calibracao, nao treinamento.** Nao treinamos modelo nenhum: rodamos o mesmo pipeline nas execucoes corretas e incorretas e comparamos as distribuicoes das metricas para escolher onde cortar o limiar.

> Por que nao o KIMORE: o RGB do KIMORE so e liberado sob pedido ao autor + EULA; o download publico dele traz so profundidade e esqueleto, incompativel com o MediaPipe.

## 11. Conectar o Google Drive e preparar o dataset

Baixe do Zenodo apenas **`videos.zip`** e **`Segmentation.csv`** e coloque numa pasta `REHAB24-6` no seu Drive (pode ignorar os zips de joints/markers). A celula abaixo monta o Drive, copia o CSV e descompacta os videos numa copia local (mais rapida que ler do Drive).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil
DRIVE = '/content/drive/MyDrive/REHAB24-6'   # ajuste se deu outro nome a pasta
REHAB = '/content/REHAB24-6'
os.makedirs(REHAB + '/videos', exist_ok=True)

assert os.path.exists(DRIVE + '/Segmentation.csv'), 'Nao achei Segmentation.csv na pasta do Drive.'
shutil.copy(DRIVE + '/Segmentation.csv', REHAB + '/Segmentation.csv')

if not os.listdir(REHAB + '/videos'):
    print('Descompactando videos.zip (1 a 2 min)...')
    with zipfile.ZipFile(DRIVE + '/videos.zip') as z:
        z.extractall(REHAB + '/videos')
print('OK. Itens na pasta de videos:', len(os.listdir(REHAB + '/videos')))

## 12. Validar a estrutura (dry-run)

Nao processa nada: so confere que cada gravacao do `Segmentation.csv` casou com um arquivo de video. Usamos `--camera Camera17` (a camera de referencia das anotacoes) para pegar uma camera so. O "ambiguo" deve ficar em 0.

In [ ]:
%cd /content/modulo_video
!python calibrar.py --raiz /content/REHAB24-6 --listar --camera Camera17

## 13. Calibrar um exercicio

Processa o exercicio 6 (**agachamento**), que envolve o corpo todo e faz as metricas de tronco e estabilidade aparecerem bem (exercicios so de braco mostrariam pouco). No fim imprime o comparativo **correto x incorreto** e sugere os limiares para o `config.py`.

> Para um teste rapido, acrescente `--max 4` (processa so 4 gravacoes). Sem ele, roda todas as gravacoes do exercicio (uns minutos por video). O MediaPipe fica silencioso enquanto processa, entao parece parado.

In [ ]:
!python calibrar.py --raiz /content/REHAB24-6 --camera Camera17 --exercise 6 --pular-frames 5

## 14. (opcional) Baixar o CSV de calibracao

O `calibracao.csv` tem as metricas por repeticao (com o rotulo correto/incorreto). Util para anexar ao relatorio.

In [ ]:
from google.colab import files
import os
if os.path.exists('data/saida/calibracao.csv'):
    files.download('data/saida/calibracao.csv')
else:
    print('Rode o passo 13 primeiro para gerar o calibracao.csv')

## 15. Integracao com o grupo (fusao multimodal)

A saida deste modulo segue o **schema comum** dos tres modulos (video / audio / clinico):

```json
{
  "patient_id": "...",
  "modulo": "video_fisioterapia",
  "tipo_anomalia": "movimento",
  "score_risco": 0.0,
  "nivel_risco": "baixo|moderado|alto",
  "descricao": "...",
  "recomendacao": "..."
}
```

Dois acordos essenciais com o grupo: **mesmo schema** de saida e **mesmo `patient_id`** entre os modulos, senao a etapa de fusao nao consegue juntar os alertas do mesmo paciente.

## Observacoes

- **CPU basta**: o gargalo (MediaPipe + YOLOv8) roda em CPU. GPU acelera pouco aqui.
- **Codec**: no Colab (Linux) o video anotado sai em H264, que toca inline no notebook. O codigo escolhe o codec automaticamente por sistema.
- **Conflito mediapipe x tensorflow (Colab)**: se aparecer `cannot import name 'runtime_version'`, va em *Ambiente de execucao > Reiniciar sessao* e rode de novo a partir do passo 2.
- **Desempenho**: para acelerar, use videos curtos/720p, ou desative o YOLOv8 com `--sem-objetos`.